# Notebook to train a classification model to identify NAMs-related project

## Import and prepare training dataset

In [2]:
from model2vec.train import StaticModelForClassification
from model2vec.inference import StaticModelPipeline

/home/mcorradi/.conda/envs/model2vec/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
import pandas as pd

df = pd.read_csv("dataset_130625.csv")

In [4]:
df.head()

,Unnamed: 0,id,Include,acronym,title,objective,frameworkProgramme,ecSignatureDate
0,1,115002,1,ETOX,Integrating bioinformatics and chemoinformatic...,The eTOX consortium proposes to develop innova...,FP7,NaN
1,2,115012,1,SAFESCIMET,European Modular Education and Training Progra...,Background:In current drug safety education an...,FP7,NaN
2,3,115336,0,MIP-DILI,Mechanism-Based Integrated Systems for the Pre...,The current test systems employed by Industry ...,FP7,NaN
3,4,115369,1,ORBITO,Oral biopharmaceutics tools,The OrBiTo project will deliver novel methods ...,FP7,NaN
4,5,115439,1,STEMBANCC,Stem cells for Biological Assays of Novel drug...,"StemBANCC, comprising internationally renowned...",FP7,NaN


In [5]:
model = StaticModelForClassification.from_pretrained()

In [6]:
from sklearn.model_selection import train_test_split

# Ensure the balance of framework programs and include/exclude is the same in training and test set
df['Include'] = df['Include'].astype(str)
df['stratify_key'] = df['Include'].astype(str) + '_' + df['frameworkProgramme'].astype(str)


# Split the dataset
train_df, test_df = train_test_split(
    df,
    test_size=0.2,
    random_state=42,
    stratify=df['stratify_key']
)

In [7]:
train_df.head()

,Unnamed: 0,id,Include,acronym,title,objective,frameworkProgramme,ecSignatureDate,stratify_key
55,56,226556,1,FMD-DISCONVAC,"Development, enhancement and complementation o...",Foot-and-mouth disease (FMD) is one of the wor...,FP7,NaN,1_FP7
20,21,210316,0,ALTERNATIVE REGIONS,Alternative Regionalisms in an Age of Globaliz...,"Recently, economic liberalization emphasis on ...",FP7,NaN,0_FP7
336,337,723951,1,IMBIBE,Innovative technology solutions to explore eff...,The human gut is host to over 100 trillion bac...,H2020,2017-10-24,1_H2020
776,777,101106199,1,CURIES,Chip URInary Engineered System for modelling b...,Urinary tract infections (UTI) are severe bact...,HORIZON,2023-04-21,1_HORIZON
706,707,101036631,1,PANORAMIX,Providing risk assessments of complex real-lif...,The toxicological impact of exposure to chemic...,H2020,2021-09-08,1_H2020


In [17]:
print(train_df['stratify_key'].value_counts())
print(test_df['stratify_key'].value_counts())
print(df['stratify_key'].value_counts())

stratify_key
0_H2020      253
1_H2020      157
0_FP7        109
1_HORIZON     81
1_FP7         53
0_HORIZON     36
Name: count, dtype: int64
stratify_key
0_H2020      64
1_H2020      40
0_FP7        27
1_HORIZON    20
1_FP7        13
0_HORIZON     9
Name: count, dtype: int64
stratify_key
0_H2020      317
1_H2020      197
0_FP7        136
1_HORIZON    101
1_FP7         66
0_HORIZON     45
Name: count, dtype: int64


## Train model

In [8]:
import time
train_df = train_df.reset_index(drop=True)
s = time.time()
model = model.fit(train_df["objective"], train_df["Include"])
print(f"training took {time.time() - s} seconds")

Seed set to 42
Using default `ModelCheckpoint`. Consider installing `litmodels` package to enable `LitModelCheckpoint` for automatic upload to the Lightning model registry.
GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
HPU available: False, using: 0 HPUs
/home/mcorradi/.conda/envs/model2vec/lib/python3.11/site-packages/lightning/pytorch/trainer/connectors/logger_connector/logger_connector.py:76: Starting from v1.9.0, `tensorboardX` has been removed as a dependency of the `lightning.pytorch` package, due to potential conflicts with other packages in the ML ecosystem. For this reason, `logger=True` will use `CSVLogger` as the default logger, unless the `tensorboard` or `tensorboardX` packages are found. Please `pip install lightning[extra]` or one of them to enable TensorBoard support by default
You are using a CUDA device ('NVIDIA A10') that has Tensor Cores. To properly utilize them, you should set `torch.set_float32_matmul_precision('medium' | 'high')

Sanity Checking: |          | 0/? [00:00<?, ?it/s]

/home/mcorradi/.conda/envs/model2vec/lib/python3.11/site-packages/lightning/pytorch/trainer/connectors/data_connector.py:425: The 'val_dataloader' does not have many workers which may be a bottleneck. Consider increasing the value of the `num_workers` argument` to `num_workers=10` in the `DataLoader` to improve performance.


/home/mcorradi/.conda/envs/model2vec/lib/python3.11/site-packages/lightning/pytorch/trainer/connectors/data_connector.py:425: The 'train_dataloader' does not have many workers which may be a bottleneck. Consider increasing the value of the `num_workers` argument` to `num_workers=10` in the `DataLoader` to improve performance.
/home/mcorradi/.conda/envs/model2vec/lib/python3.11/site-packages/lightning/pytorch/loops/fit_loop.py:310: The number of training batches (20) is smaller than the logging interval Trainer(log_every_n_steps=50). Set a lower value for log_every_n_steps if you want to see logs for the training epoch.


Epoch 0: 100%|██████████| 20/20 [00:00<00:00, 42.59it/s, v_num=0]
Validation: |          | 0/? [00:00<?, ?it/s]
Epoch 1: 100%|██████████| 20/20 [00:00<00:00, 178.67it/s, v_num=0, val_accuracy=0.841]
Validation: |          | 0/? [00:00<?, ?it/s]
Epoch 2: 100%|██████████| 20/20 [00:00<00:00, 174.55it/s, v_num=0, val_accuracy=0.855]
Validation: |          | 0/? [00:00<?, ?it/s]
Epoch 3: 100%|██████████| 20/20 [00:00<00:00, 179.00it/s, v_num=0, val_accuracy=0.870]
Validation: |          | 0/? [00:00<?, ?it/s]
Epoch 4: 100%|██████████| 20/20 [00:00<00:00, 178.16it/s, v_num=0, val_accuracy=0.841]
Validation: |          | 0/? [00:00<?, ?it/s]
Epoch 5: 100%|██████████| 20/20 [00:00<00:00, 178.46it/s, v_num=0, val_accuracy=0.855]
Validation: |          | 0/? [00:00<?, ?it/s]
Epoch 6: 100%|██████████| 20/20 [00:00<00:00, 177.98it/s, v_num=0, val_accuracy=0.870]
Validation: |          | 0/? [00:00<?, ?it/s]
Epoch 7: 100%|██████████| 20/20 [00:00<00:00, 174.25it/s, v_num=0, val_accuracy=0.884]
Val

## Calculate performance metrics on test set

### With 0.5 threshold

In [9]:
from sklearn.metrics import classification_report

predictions = model.predict(test_df["objective"])

print(classification_report(test_df["Include"], predictions))

              precision    recall  f1-score   support

           0       0.88      0.95      0.91       100
           1       0.92      0.82      0.87        73

    accuracy                           0.90       173
   macro avg       0.90      0.89      0.89       173
weighted avg       0.90      0.90      0.89       173



### With 0.95 threshold

In [11]:
import numpy as np

probs = model.predict_proba(test_df["objective"])[:, 1] 

# Apply custom threshold
threshold = 0.95
predictions2 = (probs > threshold).astype(int)
y_true = test_df["Include"].astype(int)
print(classification_report(y_true, predictions2))

              precision    recall  f1-score   support

           0       0.72      1.00      0.84       100
           1       1.00      0.48      0.65        73

    accuracy                           0.78       173
   macro avg       0.86      0.74      0.74       173
weighted avg       0.84      0.78      0.76       173



### Performance metrics on entire training set

In [13]:
import numpy as np

probs_all_training = model.predict_proba(df["objective"])[:, 1] 

# Apply custom threshold
threshold = 0.95
predictions_all_training  = (probs_all_training  > threshold).astype(int)
y_true_all_training  = df["Include"].astype(int)
print(classification_report(y_true_all_training , predictions_all_training ))

              precision    recall  f1-score   support

           0       0.84      1.00      0.91       498
           1       1.00      0.74      0.85       364

    accuracy                           0.89       862
   macro avg       0.92      0.87      0.88       862
weighted avg       0.91      0.89      0.89       862



## Predict classification for all projects

In [ ]:
all = pd.read_csv("all_projects.csv")
# Clean the input column
objectives = (
    all["objective"]
    .fillna("")              
    .astype(str)             
    .tolist()                
)
probs = model.predict_proba(objectives)
all["scores"] = probs[:, 1]
all.to_csv("all_with_predictions_13062025.csv", index=False)